# Data Mart Quality and Source Map

Gold/Silver recreation of the data quality, source mapping, and broad verification checks. Revenue is primarily `order_revenue_sgd`, excluding shipping. Legacy EDA comparisons use `order_total_incl_shipping_sgd`.


In [1]:
from pathlib import Path
import os
import warnings
warnings.filterwarnings("ignore")

_BOOT_ROOT = Path.cwd()
_MPL_DIR = (_BOOT_ROOT / "outputs" / ".matplotlib") if _BOOT_ROOT.name == "notebooks" else (_BOOT_ROOT / "notebooks" / "outputs" / ".matplotlib")
_MPL_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(_MPL_DIR))
os.environ.setdefault("XDG_CACHE_HOME", str(_MPL_DIR.parent / ".cache"))

import numpy as np
import pandas as pd
import matplotlib
if os.environ.get("NOTEBOOK_VALIDATION") == "1":
    matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    pass

try:
    from IPython.display import display, Markdown
except Exception:
    def display(x): print(x)
    def Markdown(x): return x

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = Path.cwd().resolve()

SILVER = PROJECT_ROOT / "data" / "silver"
GOLD = PROJECT_ROOT / "data" / "gold"
LEGACY_OUT = PROJECT_ROOT / "EDA" / "outputs"
NB_OUT = PROJECT_ROOT / "notebooks" / "outputs"
CHART_OUT = NB_OUT / "charts"
NB_OUT.mkdir(parents=True, exist_ok=True)
CHART_OUT.mkdir(parents=True, exist_ok=True)

ANALYSIS_DATE = pd.Timestamp("2026-04-30", tz="UTC")
MARGIN = 0.40

TEAL = "#2DC4A2"
NAVY = "#1A2E44"
SLATE = "#4A6274"
ORANGE = "#F07D3E"
RED = "#E84545"
GOLD_C = "#F7B731"
LILAC = "#9B72CF"
LIGHT_BG = "#F8F9FA"
CAT_COLORS = [TEAL, NAVY, ORANGE, GOLD_C, SLATE, RED, LILAC]
plt.rcParams.update({
    "figure.facecolor": LIGHT_BG,
    "axes.facecolor": LIGHT_BG,
    "axes.edgecolor": "#E2E8ED",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "axes.grid.axis": "y",
    "grid.color": "#E2E8ED",
    "axes.labelcolor": NAVY,
    "axes.titlecolor": NAVY,
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
    "xtick.color": SLATE,
    "ytick.color": SLATE,
    "legend.frameon": False,
    "font.family": ["DejaVu Sans"],
})

def money(x):
    return f"S${x:,.0f}"

def pct(x):
    return f"{x:.1%}"

RECREATED_CHARTS = []

def save_chart(fig, name):
    RECREATED_CHARTS.append(name)
    if os.environ.get("NOTEBOOK_VALIDATION") == "1":
        fig.canvas.draw()
    else:
        display(fig)
    plt.close(fig)
    print(f"rendered inline: {name}")
    return name

def load_marts():
    orders = pd.read_parquet(SILVER / "orders.parquet").copy()
    customers = pd.read_parquet(GOLD / "customers.parquet").copy()
    lines = pd.read_parquet(GOLD / "order_lines_enriched.parquet").copy()
    recharge_orders = pd.read_parquet(GOLD / "recharge_orders_enriched.parquet").copy()
    silver = {
        "discounts": pd.read_parquet(SILVER / "discounts.parquet"),
        "products": pd.read_parquet(SILVER / "products.parquet"),
        "campaigns": pd.read_parquet(SILVER / "campaigns.parquet"),
        "recharge_checkout": pd.read_parquet(SILVER / "recharge_checkout_items.parquet"),
        "recharge_recurring": pd.read_parquet(SILVER / "recharge_recurring_items.parquet"),
        "recharge_churned": pd.read_parquet(SILVER / "recharge_churned.parquet"),
        "recharge_reactivated": pd.read_parquet(SILVER / "recharge_reactivated.parquet"),
        "recharge_orders": pd.read_parquet(SILVER / "recharge_orders.parquet"),
    }
    orders["order_date"] = pd.to_datetime(orders["order_date"], utc=True)
    customers["first_order_date"] = pd.to_datetime(customers["first_order_date"], utc=True)
    customers["last_order_date"] = pd.to_datetime(customers["last_order_date"], utc=True)
    customers["second_order_date"] = pd.to_datetime(customers["second_order_date"], utc=True, errors="coerce")
    lines["order_date"] = pd.to_datetime(lines["order_date"], utc=True)

    orders["primary_revenue_sgd"] = pd.to_numeric(orders["order_revenue_sgd"], errors="coerce").fillna(0)
    orders["legacy_total_incl_shipping_sgd"] = pd.to_numeric(orders["order_total_incl_shipping_sgd"], errors="coerce").fillna(0)
    orders["Price: Total"] = orders["legacy_total_incl_shipping_sgd"]
    orders["Price: Total Discount"] = pd.to_numeric(orders["order_discount_sgd"], errors="coerce").fillna(0)
    orders["Price: Total Shipping"] = pd.to_numeric(orders["shipping_revenue_sgd"], errors="coerce").fillna(0)
    orders["revenue_basis_delta_sgd"] = orders["legacy_total_incl_shipping_sgd"] - orders["primary_revenue_sgd"]

    customers["total_revenue"] = pd.to_numeric(customers["total_revenue_sgd"], errors="coerce").fillna(0)
    customers["total_discount"] = pd.to_numeric(customers["total_discount_sgd"], errors="coerce").fillna(0)
    customers["first_product_cat"] = customers["first_product_category"]
    customers["is_repeat"] = customers["is_repeat"].astype(bool)

    for col in ["Line: Quantity", "Line: Price", "Line: Discount", "Line: Total"]:
        if col in lines.columns:
            lines[col] = pd.to_numeric(lines[col], errors="coerce").fillna(0)
    return orders, customers, lines, recharge_orders, silver

orders, customers, lines, recharge_orders, silver = load_marts()
print(f"Loaded marts: {len(orders):,} orders, {len(customers):,} customers, {len(lines):,} enriched lines")
print("Primary revenue excludes shipping. Legacy comparability uses order_total_incl_shipping_sgd.")


Loaded marts: 27,350 orders, 13,780 customers, 50,963 enriched lines
Primary revenue excludes shipping. Legacy comparability uses order_total_incl_shipping_sgd.


## Mart Inventory and Sanity Checks


In [2]:
mart_paths = {
    "silver_orders": SILVER / "orders.parquet",
    "silver_lines": SILVER / "lines.parquet",
    "silver_non_product_lines": SILVER / "order_non_product_lines.parquet",
    "silver_products": SILVER / "products.parquet",
    "silver_discounts": SILVER / "discounts.parquet",
    "silver_campaigns": SILVER / "campaigns.parquet",
    "silver_recharge_orders": SILVER / "recharge_orders.parquet",
    "silver_recharge_checkout": SILVER / "recharge_checkout_items.parquet",
    "silver_recharge_recurring": SILVER / "recharge_recurring_items.parquet",
    "silver_recharge_churned": SILVER / "recharge_churned.parquet",
    "silver_recharge_reactivated": SILVER / "recharge_reactivated.parquet",
    "gold_customers": GOLD / "customers.parquet",
    "gold_order_lines_enriched": GOLD / "order_lines_enriched.parquet",
    "gold_recharge_orders_enriched": GOLD / "recharge_orders_enriched.parquet",
}
rows = []
for name, path in mart_paths.items():
    df = pd.read_parquet(path)
    rows.append({"mart": name, "rows": len(df), "columns": len(df.columns), "path": str(path.relative_to(PROJECT_ROOT))})
inventory = pd.DataFrame(rows)
display(inventory)
inventory.to_csv(NB_OUT / "00_mart_inventory.csv", index=False)

checks = pd.DataFrame([
    {"check": "orders row count", "value": len(orders), "expected": 27350, "pass": len(orders) == 27350},
    {"check": "customers row count", "value": len(customers), "expected": 13780, "pass": len(customers) == 13780},
    {"check": "enriched line row count", "value": len(lines), "expected": 50963, "pass": len(lines) == 50963},
    {"check": "customers match unique order customers", "value": customers.customer_id.nunique(), "expected": orders.customer_id.nunique(), "pass": customers.customer_id.nunique() == orders.customer_id.nunique()},
    {"check": "all orders have nonnegative primary revenue", "value": int((orders.primary_revenue_sgd >= 0).sum()), "expected": len(orders), "pass": bool((orders.primary_revenue_sgd >= 0).all())},
])
display(checks)
checks.to_csv(NB_OUT / "00_sanity_checks.csv", index=False)


,mart,rows,columns,path
0,silver_orders,27350,49,data/silver/orders.parquet
1,silver_lines,50963,42,data/silver/lines.parquet
2,silver_non_product_lines,102608,41,data/silver/order_non_product_lines.parquet
3,silver_products,56,25,data/silver/products.parquet
4,silver_discounts,367,17,data/silver/discounts.parquet
5,silver_campaigns,137033,10,data/silver/campaigns.parquet
6,silver_recharge_orders,1215,10,data/silver/recharge_orders.parquet
7,silver_recharge_checkout,1094,14,data/silver/recharge_checkout_items.parquet
8,silver_recharge_recurring,650,14,data/silver/recharge_recurring_items.parquet
9,silver_recharge_churned,526,12,data/silver/recharge_churned.parquet


,check,value,expected,pass
0,orders row count,27350,27350,True
1,customers row count,13780,13780,True
2,enriched line row count,50963,50963,True
3,customers match unique order customers,13780,13780,True
4,all orders have nonnegative primary revenue,27350,27350,True


## Core Business Summaries


In [3]:
orders_y = orders.assign(year=orders.order_date.dt.year)
annual = (
    orders_y.groupby("year")
    .agg(
        orders=("order_id", "count"),
        customers=("customer_id", "nunique"),
        revenue_sgd=("primary_revenue_sgd", "sum"),
        legacy_revenue_incl_shipping_sgd=("legacy_total_incl_shipping_sgd", "sum"),
        shipping_delta_sgd=("revenue_basis_delta_sgd", "sum"),
        discount_sgd=("Price: Total Discount", "sum"),
        discount_orders=("has_discount", "sum"),
    )
    .assign(
        avg_order_value=lambda d: d.revenue_sgd / d.orders,
        disc_rate=lambda d: d.discount_sgd / d.revenue_sgd.replace(0, np.nan),
        disc_pct_orders=lambda d: d.discount_orders / d.orders,
        legacy_delta_pct=lambda d: d.shipping_delta_sgd / d.legacy_revenue_incl_shipping_sgd.replace(0, np.nan),
    )
)
display(annual)
annual.to_csv(NB_OUT / "00_orders_by_year_gold_basis.csv")

monthly = (
    orders.assign(month=orders.order_date.dt.to_period("M").astype(str))
    .groupby("month")
    .agg(orders=("order_id", "count"), revenue_sgd=("primary_revenue_sgd", "sum"), legacy_revenue_incl_shipping_sgd=("legacy_total_incl_shipping_sgd", "sum"))
)
monthly.to_csv(NB_OUT / "00_revenue_by_month_gold_basis.csv")

top_country = orders.groupby("Shipping: Country").agg(orders=("order_id", "count"), revenue_sgd=("primary_revenue_sgd", "sum")).sort_values("orders", ascending=False).head(10)
channel = orders.groupby("channel").agg(orders=("order_id", "count"), customers=("customer_id", "nunique"), revenue_sgd=("primary_revenue_sgd", "sum"), discount_sgd=("Price: Total Discount", "sum")).assign(aov=lambda d: d.revenue_sgd / d.orders, disc_rate=lambda d: d.discount_sgd / d.revenue_sgd.replace(0, np.nan)).sort_values("orders", ascending=False)
display(top_country)
display(channel)
top_country.to_csv(NB_OUT / "00_orders_by_country_gold_basis.csv")
channel.to_csv(NB_OUT / "00_orders_by_channel_gold_basis.csv")


,orders,customers,revenue_sgd,legacy_revenue_incl_shipping_sgd,shipping_delta_sgd,discount_sgd,discount_orders,avg_order_value,disc_rate,disc_pct_orders,legacy_delta_pct
year,,,,,,,,,,,
2019,3,3,98.127273,108.481818,10.354545,0.000000,0,32.709091,0.000000,0.000000,0.095450
2020,2847,1696,449820.209697,456952.094545,7131.884848,0.000000,0,157.997966,0.000000,0.000000,0.015608
2021,6259,3815,838693.811212,847930.017273,9236.206061,0.000000,0,133.998053,0.000000,0.000000,0.010893
2022,3710,2299,740879.937879,747586.813939,6706.876061,30072.956970,563,199.698096,0.040591,0.151752,0.008971
2023,2253,1399,179056.153636,182037.636667,2981.483030,32418.838485,1337,79.474547,0.181054,0.593431,0.016378
2024,4261,2621,280449.056667,284970.730606,4521.673939,127407.735455,2950,65.817662,0.454299,0.692326,0.015867
2025,6407,4107,404128.928619,409151.528316,5022.599697,222033.176150,3181,63.076156,0.549412,0.496488,0.012276
2026,1610,1230,182613.100000,184214.480000,1601.380000,52251.520000,993,113.424286,0.286132,0.616770,0.008693


,orders,revenue_sgd
Shipping: Country,,
Malaysia,14278,1.370471e+06
Singapore,11345,1.551848e+06
Hong Kong,308,1.886738e+04
Indonesia,263,1.068907e+04
Japan,64,9.789000e+02
Australia,19,1.547850e+03
Philippines,5,3.891000e+02
United Kingdom,5,5.143600e+02
United States,4,1.758300e+02


,orders,customers,revenue_sgd,discount_sgd,aov,disc_rate
channel,,,,,,
Direct / Organic,13208,7252,1.352612e+06,267625.767665,102.408522,0.197859
Subscription,10230,4584,1.449163e+06,121901.998788,141.658201,0.084119
Marketplace,3268,2126,2.430273e+05,71559.452121,74.365764,0.294450
Paid Social,473,442,2.053755e+04,659.657879,43.419757,0.032120
Email,128,123,7.448130e+03,1557.540606,58.188516,0.209118
Affiliate,40,34,2.801480e+03,879.810000,70.037000,0.314052
Paid Search,3,3,1.497000e+02,0.000000,49.900000,0.000000


## Product, Discount, Campaign, and Lineage Notes


In [4]:
products = silver["products"]
discounts = silver["discounts"]
campaigns = silver["campaigns"]
product_summary = pd.DataFrame([
    {"metric": "silver product variants", "value": len(products)},
    {"metric": "active product variants", "value": int((products.get("status", pd.Series(dtype=str)) == "active").sum()) if "status" in products else np.nan},
    {"metric": "discount codes", "value": len(discounts)},
    {"metric": "discount codes with usage", "value": int((pd.to_numeric(discounts["Times Used In Total"], errors="coerce").fillna(0) > 0).sum())},
    {"metric": "campaign rows", "value": len(campaigns)},
])
display(product_summary)

lineage = pd.DataFrame([
    {"legacy_source": "EDA/01_load_and_merge.py", "gold_silver_replacement": "data_cleaning notebooks and data/silver + data/gold marts", "verification_note": "Raw import logic is downstream of current marts and is not recreated here."},
    {"legacy_source": "EDA/02_data_quality.py", "gold_silver_replacement": "This notebook", "verification_note": "Recreated using mart columns; product master row count differs because Silver is cleaned."},
    {"legacy_source": "EDA/trace_datasets.py", "gold_silver_replacement": "Inventory and lineage tables here", "verification_note": "Raw Excel/CSV row traces are partial unless raw files are re-read."},
    {"legacy_source": "EDA/verify_all.py", "gold_silver_replacement": "Sanity checks and notebook-specific validation tables", "verification_note": "Recomputed against Gold/Silver outputs."},
])
display(lineage)
product_summary.to_csv(NB_OUT / "00_product_discount_campaign_summary.csv", index=False)
lineage.to_csv(NB_OUT / "00_source_map.csv", index=False)


,metric,value
0,silver product variants,56
1,active product variants,31
2,discount codes,367
3,discount codes with usage,163
4,campaign rows,137033


,legacy_source,gold_silver_replacement,verification_note
0,EDA/01_load_and_merge.py,data_cleaning notebooks and data/silver + data...,Raw import logic is downstream of current mart...
1,EDA/02_data_quality.py,This notebook,Recreated using mart columns; product master r...
2,EDA/trace_datasets.py,Inventory and lineage tables here,Raw Excel/CSV row traces are partial unless ra...
3,EDA/verify_all.py,Sanity checks and notebook-specific validation...,Recomputed against Gold/Silver outputs.
